## Qlora : 
&nbsp;&nbsp;&nbsp; In this notebook, we try to implement Qlora to finetune the desired llm **Qwen2.5-Coder-7B-Instruct** using Qlora, and by using unsloth api. The purpose behind this notebook is to have a llm finetuned on a very limited resources. 

In [12]:

!pip install unsloth
!pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes datasets transformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-37n9rqr6/unsloth_4026f0249c764b5d9c7a527853d289a7
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-37n9rqr6/unsloth_4026f0249c764b5d9c7a527853d289a7
  Resolved https://github.com/unslothai/unsloth.git to commit 53af4a1b3e78f2d1cac9401fe15baf8720fc2303
  Installing build dependencies ... one
  Getting requirements to build wheel ... one
  Preparing metadata (pyproject.toml) ... one


In [16]:
! conda install -y gcc -c conda-forge

Retrieving ndone
Channels:
 - conda-forge
Platform: linux-64
doneecting package metadata (repodata.json): - 
doneing environment: \ 


==> WARNING: A newer version of conda exists. <==
    current version: 25.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - gcc


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    binutils_impl_linux-64-2.43|       h4bf12b8_2         5.4 MB  conda-forge
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    certifi-2026.2.25          |     pyhd8ed1ab_0         148 KB  conda-forge
    gcc-14.2.0                 |       h96c4ede_2          54 KB  conda-forge
    gcc_impl_linux-64-14.2.0   |       hdb7739f_2        70.1 MB  conda-forge
    kernel-headers_linux-64-5.14.0|       he073ed

In [2]:
from unsloth import FastLanguageModel
import torch
import os 
max_seq_length = 2048   
dtype = None           
load_in_4bit = True  # Qlora uses the quantized model rather than the original base model    
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=2048,
    dtype = dtype, 
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 NVL MIG 1g.24gb. Num GPUs = 1. Max memory: 21.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                         
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,               
    bias = "none",               
    use_gradient_checkpointing = "unsloth",  
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

model.print_trainable_parameters()

Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [5]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset("SakanaAI/AI-CUDA-Engineer-Archive")

splits = [
    dataset["level_1"].filter(lambda x: x["Correct"] == True),
    dataset["level_2"].filter(lambda x: x["Correct"] == True),
    dataset["level_3"].filter(lambda x: x["Correct"] == True),
]
dataset = concatenate_datasets(splits)


# pick 20 samples to do a health check on the pipeline 
dataset = dataset.select(range(20))

print(f"Training on {len(dataset)} samples")

Training on 20 samples


### Remark : 
Downbelow, we will set the data in the structure that would be passed for training , for the sampling strategies either bucket or island, the method will be altered later on to do so, we will need to rely on a metric to do the bucketing , we will try to apply a real simple one. 

In [7]:
cuda_rl_task = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are an expert CUDA kernel engineer. Given a PyTorch module implementation, write an optimized CUDA kernel that is functionally equivalent but achieves maximum GPU performance.

Operation: {Op_Name}

### Input:
{PyTorch_Code_Module}

### Response:
{CUDA_Code}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for op, pytorch_code, cuda_code in zip(
        examples["Op_Name"],
        examples["PyTorch_Code_Module"],
        examples["CUDA_Code"],
    ):
        text = cuda_rl_task.format(
            Op_Name              = op,
            PyTorch_Code_Module  = pytorch_code,
            CUDA_Code            = cuda_code,
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Total training samples: {len(dataset)}")
print(dataset[0]["text"][:500])

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Total training samples: 20
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are an expert CUDA kernel engineer. Given a PyTorch module implementation, write an optimized CUDA kernel that is functionally equivalent but achieves maximum GPU performance.

Operation: 1_Square_matrix_multiplication_

### Input:
import torch
import torch.nn as nn


class Model(nn.Module):
    """
    Simple model that performs a single square matrix multiplication (C


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,   
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,                          
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

num_proc must be <= 20. Reducing num_proc to 20 for dataset of size 20.
[datasets.arrow_dataset|WARNING]num_proc must be <= 20. Reducing num_proc to 20 for dataset of size 20.


Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/20 [00:00<?, ? examples/s]

In [9]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}  |  Total VRAM: {max_memory} GB  |  Reserved: {start_gpu_memory} GB")

GPU: NVIDIA H100 NVL MIG 1g.24gb  |  Total VRAM: 21.625 GB  |  Reserved: 5.393 GB


In [18]:
import warnings
warnings.filterwarnings("ignore", message="Unable to fetch remote file")
warnings.filterwarnings("ignore", message="Could not find a config file")

In [19]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 1 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
